# Valutazioni finali di stage – analisi per il management

Notebook riproducibile dall'inizio alla fine. Ogni step è separato da un'intestazione;
i trattamenti sui dati vengono applicati **solo dopo conferma esplicita** e sono documentati
nel *Registro assunzioni e limiti* in fondo a ogni step.

**Input:** `data/INT_VAL_ASSESSMENT_SET.csv` (non versionato nel repo)
**Output:** tabelle in `outputs/tables/`, grafici in `outputs/figures/`

## Step 0 – Struttura del file (nessuna analisi, nessun trattamento)

**Approccio.** Il file viene letto con *tutte le colonne come testo* (`dtype=str`, `keep_default_na=False`).

*Alternative considerate:*
- lettura con i default di pandas: scartata, perché converte in silenzio `"N/A"` e le celle vuote in `NaN`
  e assegna tipi diversi a colonne omogenee (es. `Listening` diventa testo a causa di un solo valore `BBBB`),
  nascondendo proprio le anomalie che vogliamo vedere;
- lettura dall'`.xlsx`: usato solo come controllo incrociato (deve essere identico al CSV).

Così ogni valore viene visto esattamente come scritto nel file; tipi e anomalie sono *diagnosticati*, non corretti.

In [1]:
# --- Import e percorsi -------------------------------------------------------
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

DATA_CSV  = Path("data/INT_VAL_ASSESSMENT_SET.csv")
DATA_XLSX = Path("data/INT_VAL_ASSESSMENT_SET.xlsx")   # opzionale, solo per controllo incrociato
OUT_TAB   = Path("outputs/tables"); OUT_TAB.mkdir(parents=True, exist_ok=True)

# Colonne attese secondo la traccia (da verificare, non da dare per scontate)
SKILLS = ["Digital", "Teamwork", "Entrepreneurial Mindset", "Critical Thinking", "Creativity",
          "Planning", "Flexibility", "Cultural Awareness", "Writing", "Listening",
          "Time Management", "Responsibility", "Motivation", "Feedback Management",
          "Interpersonal Skills", "Leadership"]
SCORES = SKILLS + ["OVERALL"]                      # 17 colonne su scala attesa 1-10
CATEG  = ["NAZIONE", "SETTORE AZIENDALE", "AREA FUNZIONALE", "CORSO", "Hiring"]
EXPECTED_COLS = ["ID", "DURATA", "NAZIONE", "SETTORE AZIENDALE", "AREA FUNZIONALE", "CORSO"] + SCORES + ["Hiring"]

### 0.1 Controlli sul file grezzo (byte e righe)
Prima di usare pandas guardiamo il file "a mano": codifica, separatore, fine riga, numero di campi per riga.

In [2]:
raw = DATA_CSV.read_bytes()

# Codifica: se non è UTF-8 valido, cerchiamo i byte non ASCII per capire quale codifica usare
try:
    raw.decode("utf-8"); ENCODING = "utf-8"
except UnicodeDecodeError:
    ENCODING = "cp1252"   # nessun byte 0x80-0x9F -> cp1252 e latin-1 coincidono su questo file
non_ascii = Counter(b for b in raw if b >= 0x80)
print("Codifica usata:", ENCODING, "| byte non ASCII:", {hex(k): v for k, v in non_ascii.items()})
print("Esempio decodificato:", bytes([0xE0]).decode(ENCODING))

text  = raw.decode(ENCODING)
lines = [l for l in text.split("\r\n") if l != ""]
print("Fine riga CRLF:", "\r\n" in text)
print("Righe non vuote (incluso header):", len(lines))
print("Numero di ';' per riga:", Counter(l.count(";") for l in lines))

# Controlli automatici sulla struttura grezza
assert all(l.count(";") == 23 for l in lines), "righe con numero di campi diverso da 24"
N_ROWS_RAW = len(lines) - 1          # righe dati, header escluso

Codifica usata: cp1252 | byte non ASCII: {'0xe0': 634}
Esempio decodificato: à
Fine riga CRLF: True
Righe non vuote (incluso header): 4383
Numero di ';' per riga: Counter({23: 4383})


### 0.2 Caricamento (tutto come testo) e verifica delle colonne

In [3]:
df_raw = pd.read_csv(DATA_CSV, sep=";", encoding=ENCODING, dtype=str, keep_default_na=False)
print("Shape:", df_raw.shape)

# Verifica indipendente: righe lette da pandas == righe contate sul file grezzo
assert len(df_raw) == N_ROWS_RAW, (len(df_raw), N_ROWS_RAW)
# Verifica colonne attese: stessi nomi, stesso ordine, nessuno spazio spurio
assert list(df_raw.columns) == EXPECTED_COLS, set(df_raw.columns) ^ set(EXPECTED_COLS)
assert all(c == c.strip() for c in df_raw.columns)
print("Colonne verificate:", len(df_raw.columns))

Shape: (4382, 24)
Colonne verificate: 24


### 0.3 Profilo delle colonne
Per ogni colonna: valori distinti, celle vuote, `N/A` letterali, quanti valori sono numeri interi,
token non numerici, min/max dei valori numerici e (per i punteggi) quanti cadono fuori dalla scala 1-10.
Il "tipo diagnosticato" è quello che la colonna *dovrebbe* avere; il "tipo pandas default" mostra cosa
succederebbe con una lettura ingenua.

In [4]:
df_default = pd.read_csv(DATA_CSV, sep=";", encoding=ENCODING)   # solo per confronto dei tipi

rows = []
for c in df_raw.columns:
    s   = df_raw[c]
    num = pd.to_numeric(s, errors="coerce")                         # NaN dove non numerico
    tok = s[num.isna() & (s != "") & (s != "N/A")]                  # token anomali (né vuoto né N/A)
    is_num_col = num.notna().mean() > 0.9
    rows.append({
        "colonna": c,
        "tipo_pandas_default": str(df_default[c].dtype),
        "tipo_diagnosticato": (("intero, atteso 1-10" if c in SCORES else "intero") if is_num_col else "categorica (testo)"),
        "n_righe": len(s),
        "n_distinti": s.nunique(),
        "n_vuoti": int((s == "").sum()),
        "n_NA_letterale": int((s == "N/A").sum()),
        "n_numerici": int(num.notna().sum()),
        "token_anomali": {k: int(v) for k, v in tok.value_counts().items()} if is_num_col and len(tok) else "",
        "min": num.min() if is_num_col else "",
        "max": num.max() if is_num_col else "",
        "n_fuori_1_10": int(((num < 1) | (num > 10)).sum()) if c in SCORES else "",
        "n_spazi_iniziali_finali": int((s != s.str.strip()).sum()),
    })
profilo = pd.DataFrame(rows)
profilo.to_csv(OUT_TAB / "step0_profilo_colonne.csv", index=False)
profilo

,colonna,tipo_pandas_default,tipo_diagnosticato,n_righe,n_distinti,n_vuoti,n_NA_letterale,n_numerici,token_anomali,min,max,n_fuori_1_10,n_spazi_iniziali_finali
0,ID,int64,intero,4382,4382,0,0,4382,,1,4388,,0
1,DURATA,int64,intero,4382,247,0,0,4382,,20,45534,,0
2,NAZIONE,str,categorica (testo),4382,65,3,0,0,,,,,0
3,SETTORE AZIENDALE,str,categorica (testo),4382,38,0,0,0,,,,,131
4,AREA FUNZIONALE,str,categorica (testo),4382,23,1,0,0,,,,,0
5,CORSO,str,categorica (testo),4382,38,1,0,0,,,,,0
6,Digital,float64,"intero, atteso 1-10",4382,12,1,65,4316,,1.0,10.0,0,0
7,Teamwork,float64,"intero, atteso 1-10",4382,12,1,90,4291,,1.0,10.0,0,0
8,Entrepreneurial Mindset,float64,"intero, atteso 1-10",4382,13,1,153,4228,,1.0,7000.0,1,0
9,Critical Thinking,float64,"intero, atteso 1-10",4382,12,1,127,4254,,1.0,10.0,0,0


### 0.4 Variabili categoriche: valori distinti e conteggi
I valori sono mostrati con `repr()` per rendere visibili celle vuote (`''`) e spazi finali.

In [5]:
vc_list = []
for c in CATEG:
    vc = df_raw[c].value_counts(dropna=False)
    vc_list.append(pd.DataFrame({"variabile": c, "valore": vc.index, "n": vc.values,
                                 "pct": (vc.values / len(df_raw) * 100).round(2)}))
    # Verifica: la somma dei conteggi deve dare il totale righe
    assert vc.sum() == len(df_raw)
valori_cat = pd.concat(vc_list, ignore_index=True)
valori_cat.to_csv(OUT_TAB / "step0_valori_categoriche.csv", index=False)

for c in CATEG:
    sub = valori_cat[valori_cat.variabile == c]
    print(f"\n=== {c}: {len(sub)} valori distinti (n totale = {sub.n.sum()}) ===")
    print(sub.assign(valore=sub.valore.map(repr))[["valore", "n", "pct"]].to_string(index=False))


=== NAZIONE: 65 valori distinti (n totale = 4382) ===
                            valore    n   pct
                          'ITALIA' 3403 77.66
                         'FRANCIA'  157  3.58
                        'GERMANIA'  114  2.60
                          'BELGIO'   87  1.99
'GRAN BRETAGNA E IRLANDA DEL NORD'   81  1.85
                     'LUSSEMBURGO'   63  1.44
                        'SVIZZERA'   63  1.44
           "STATI UNITI D'AMERICA"   48  1.10
        'CINA REPUBBLICA POPOLARE'   36  0.82
                          'SPAGNA'   33  0.75
                         'TURCHIA'   26  0.59
                     'PAESI BASSI'   25  0.57
           'REPUBBLICA DI POLONIA'   21  0.48
                         'ROMANIA'   18  0.41
                       'SINGAPORE'   17  0.39
             'EMIRATI ARABI UNITI'   16  0.37
                          'GRECIA'   14  0.32
                       'DANIMARCA'   10  0.23
                         'MAROCCO'   10  0.23
                      'PO

### 0.5 Hiring – valori presenti (con verifica indipendente)

In [6]:
hiring_vc = df_raw["Hiring"].value_counts()
print(hiring_vc.rename(index=repr))

# Verifica indipendente: filtro semplice vs value_counts
for v in hiring_vc.index:
    assert (df_raw["Hiring"] == v).sum() == hiring_vc[v]
print("\nRighe con Hiring diverso da SI/NO:")
print(df_raw.loc[~df_raw["Hiring"].isin(["SI", "NO"]), ["ID", "OVERALL", "Hiring"]])

Hiring
'SI'      2494
'NO'      1885
''           2
'Null'       1
Name: count, dtype: int64

Righe con Hiring diverso da SI/NO:
        ID OVERALL Hiring
97      98      10   Null
813    816               
3852  3859      10       


### 0.6 ID e DURATA

In [7]:
ids = df_raw["ID"].astype(int)        # sicuro: verificato in 0.3 che è sempre intero
mancanti = sorted(set(range(1, ids.max() + 1)) - set(ids))
print(f"ID: n={len(ids)}, distinti={ids.nunique()}, min={ids.min()}, max={ids.max()}, "
      f"ordinati={ids.is_monotonic_increasing}, buchi nella sequenza={mancanti}")
assert ids.is_unique

dur = df_raw["DURATA"].astype(int)
print("\nDURATA (giorni) – statistiche descrittive, n =", dur.notna().sum())
print(dur.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(1).to_string())

# Valori molto grandi: sono plausibili come seriali di data Excel? (solo diagnosi, nessuna correzione)
grandi = df_raw.loc[dur > 400, ["ID", "DURATA", "NAZIONE", "CORSO"]].copy()
grandi["se_fosse_data_excel"] = pd.to_datetime(grandi["DURATA"].astype(int), unit="D",
                                               origin="1899-12-30").dt.date
grandi

ID: n=4382, distinti=4382, min=1, max=4388, ordinati=True, buchi nella sequenza=[266, 378, 1918, 2500, 3093, 3443]

DURATA (giorni) – statistiche descrittive, n = 4382
count     4382.0
mean       179.9
std       1812.5
min         20.0
1%          30.0
5%          39.0
25%         60.0
50%         91.0
75%        171.0
95%        182.0
99%        344.8
max      45534.0


,ID,DURATA,NAZIONE,CORSO,se_fosse_data_excel
17,18,547,SERBIA,B12,1901-06-30
106,107,45303,FRANCIA,B03,2024-01-12
507,510,45394,ITALIA,B16,2024-04-12
940,943,45485,ITALIA,B01,2024-07-12
1079,1082,45504,ITALIA,B01,2024-07-31
1282,1285,45492,ITALIA,B10,2024-07-19
1891,1894,45527,ITALIA,B11,2024-08-23
2302,2306,729,ITALIA,B17,1901-12-29
3139,3145,45534,GRAN BRETAGNA E IRLANDA DEL NORD,T07,2024-08-30


### 0.7 Anomalie puntuali e duplicati (solo elenco, nessuna correzione)

In [8]:
num_scores = df_raw[SCORES].apply(pd.to_numeric, errors="coerce")

# Righe con almeno una cella vuota
righe_vuote = df_raw[(df_raw == "").any(axis=1)]
print(f"Righe con almeno una cella vuota: n={len(righe_vuote)}")
print(righe_vuote[["ID", "DURATA", "NAZIONE", "AREA FUNZIONALE", "CORSO", "OVERALL", "Hiring"]], "\n")

# Punteggi fuori scala (numerici fuori 1-10) o token non numerici diversi da N/A
fuori = (num_scores < 1) | (num_scores > 10)
token = num_scores.isna() & ~df_raw[SCORES].isin(["", "N/A"])
anom = df_raw.loc[(fuori | token).any(axis=1), ["ID"] + SCORES]
print(f"Righe con punteggi fuori scala o token anomali: n={len(anom)}")
print(anom.to_string(index=False), "\n")

# N/A letterali per riga
na_per_riga = (df_raw[SCORES] == "N/A").sum(axis=1)
print("Distribuzione n. di N/A per riga:", na_per_riga.value_counts().sort_index().to_dict())
print("Righe con almeno un N/A:", int((na_per_riga > 0).sum()), "\n")

# Duplicati: identici su tutte le colonne tranne ID
dup_mask = df_raw.drop(columns="ID").duplicated(keep=False)
print(f"Duplicati esatti (escluso ID): righe coinvolte={dup_mask.sum()}, "
      f"copie in eccesso={df_raw.drop(columns='ID').duplicated().sum()}")
print("di cui con tutti i 17 punteggi = 10:", int((df_raw.loc[dup_mask, SCORES] == "10").all(axis=1).sum()))
print("Righe totali con tutti i 17 punteggi = 10:", int((df_raw[SCORES] == "10").all(axis=1).sum()))

pd.concat([
    anom.assign(tipo="punteggio fuori scala / token")[["ID", "tipo"]],
    righe_vuote.assign(tipo="cella vuota")[["ID", "tipo"]],
    df_raw.loc[dup_mask].assign(tipo="duplicato escluso ID")[["ID", "tipo"]],
    df_raw.loc[dur > 400].assign(tipo="DURATA > 400")[["ID", "tipo"]],
]).sort_values("ID").to_csv(OUT_TAB / "step0_righe_anomale.csv", index=False)

Righe con almeno una cella vuota: n=4
        ID DURATA NAZIONE                       AREA FUNZIONALE CORSO OVERALL Hiring
813    816    330                                                                   
1131  1134    212                            Ricerca & Sviluppo   B01      10     NO
2027  2031     85                       Attività polifunzionali   T01      10     SI
3852  3859    195  ITALIA  Amministrazione, Finanza e Controllo   M03      10        

Righe con punteggi fuori scala o token anomali: n=8
  ID Digital Teamwork Entrepreneurial Mindset Critical Thinking Creativity Planning Flexibility Cultural Awareness Writing Listening Time Management Responsibility Motivation Feedback Management Interpersonal Skills Leadership OVERALL
   1      10       10                      10                10         10       10          10                  A      10        10              10             10         10                  10                   10         10      10
  42      10 

### 0.8 Verifica indipendente: CSV vs XLSX

In [9]:
if DATA_XLSX.exists():
    xl = pd.read_excel(DATA_XLSX, sheet_name=None, dtype=str, keep_default_na=False)
    print("Fogli:", {k: v.shape for k, v in xl.items()})
    x = next(iter(xl.values()))
    assert list(x.columns) == list(df_raw.columns) and x.shape == df_raw.shape
    n_diff = int((x.values != df_raw.values).sum())
    print("Celle diverse tra CSV e XLSX:", n_diff)
    assert n_diff == 0
else:
    print("XLSX non presente: controllo saltato")

Fogli: {'in': (4382, 24)}
Celle diverse tra CSV e XLSX: 0


### Registro assunzioni e limiti – aggiornato a fine Step 0

| # | Tipo | Voce | Stato |
|---|------|------|-------|
| A1 | Assunzione | Codifica `cp1252` (unico byte non ASCII: `0xE0` = "à"); separatore `;`; fine riga CRLF. | verificata |
| A2 | Assunzione | `NAZIONE` = paese della sede dello stage, non nazionalità dello studente (da brief). | da brief |
| A3 | Assunzione | `CORSO` trattato solo come codice anonimo; nessuna ipotesi sul tipo di corso. | da brief |
| A4 | Assunzione | Scala attesa dei punteggi: interi 1-10. | da brief, coerente con 99,9% dei valori |
| L1 | Qualità | `N/A` letterale in 15 skill su 16 (non in `Leadership` né in `OVERALL`); 470 righe con almeno un N/A. Significato (non osservato? non applicabile?) ignoto. | da decidere |
| L2 | Qualità | 7 valori fuori scala (7000, 88, 999, 55, 100) e 3 token non numerici (`A`, `BBBB`, `NN`). | da decidere |
| L3 | Qualità | Righe con celle vuote: ID 816 (quasi vuota), 1134 e 2031 (NAZIONE vuota), 3859 (Hiring vuoto); ID 98 ha Hiring = `Null`. | da decidere |
| L4 | Qualità | `DURATA`: 7 valori tra 45.303 e 45.534 compatibili con date Excel 2024; 2 valori (547, 729) alti ma non impossibili. | da decidere |
| L5 | Qualità | 9 coppie identiche su tutte le colonne tranne ID; 8 su 9 sono valutazioni "tutto 10", che nel file sono 742: potrebbero essere coincidenze. | da decidere |
| L6 | Qualità | `SETTORE AZIENDALE` 'ONG ed Enti No Profit ' ha uno spazio finale (unica categoria, nessuno sdoppiamento). | da decidere (cosmetico) |
| L7 | Qualità | `CORSO` = 'L' (n=314) ha formato diverso dagli altri codici (lettera + 2 cifre). | da chiarire |
| L8 | Limite | Forte sbilanciamento: ITALIA 3.403 righe su 4.382; molti paesi, corsi e aree con n molto piccolo. | da gestire in analisi |
| L9 | Limite | ID non contigui (mancano 6 ID): possibile pre-filtraggio a monte, ma nessun duplicato di ID. | informativo |